# Libs & Setup

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import scale, inverse_scale, inverse_scale_pair, inspect, monte_carlo_statistic
from utils.paths import CHECKPOINTS_DIR, REPORTS_SIM_DIR, RESULTS_DIR, REPORTS_QS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

# Own Libs
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

/home/narodom.y@FUSION.LAB/.conda/envs/tsgenai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

In [3]:
cfg = TrainConfig(epochs=2, window_size=64, device=torch.device("cuda:0"))
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
batch_size_exp = 1
epochs = cfg.epochs
sim_steps = cfg.steps_to_sim
num_sims = cfg.num_sims

# Optimizer
weight_decay = cfg.optimizer.weight_decay
lr = cfg.optimizer.lr
stride = 5
time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 32,
    'dim_feedforward': 512,
    'dropout': 0.1
}

ckpt_name = f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}"

In [4]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")
    
    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

# Data

In [5]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

In [6]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

Data range: 2015-01-02 00:00:00 to 2025-12-22 00:00:00
Total duration: 4007 days 00:00:00
Data range: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Total duration: 1457 days 00:00:00


In [7]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [8]:
df = basket.data
df.ffill(inplace=True)
df.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                  TSLA                                                 ...  \
                 Close        High         Low        Open     Volume  ...   
Date                                                                   ...   
2021-01-04  243.256668  248.163330  239.063339  239.820007  145914600  ...   
2021-01-05  245.036667  246.946671  239.733337  241.220001   96735600  ...   
2021-01-06  251.993332  258.000000  249.699997  252.830002  134100000  ...   
2021-01-07  272.013336  272.329987  258.399994  259.209991  154496700  ...   
2021-01-08  293.339996  294.829987  279.463318  285.333344  225166500  ...   

                  AMD                                                  CSCO  \
                Close       High        Low       Open    Volume      Close   
Date                                                                          
2021-01-04  92.300003  96.059998  90.919998  92.110001  51802600  38.239326   
2021-01-05  92.769997  93.209999  91.410004  92.099998  34208000  38.256729   
2021-01-06  90.330002  92.279999  89.459999  91.620003  51911700  38.622074   
2021-01-07  95.160004  95.510002  91.199997  91.330002  42897200  39.109196   
2021-01-08  94.580002  96.400002  93.269997  95.980003  39816400  39.196186   

                                                       
                 High        Low       Open    Volume  
Date                                                   
2021-01-04  38.595972  37.708707  38.543782  24392500  
2021-01-05  38.335017  37.734811  37.995770  17763700  
2021-01-06  39.030909  38.178440  38.387210  21823100  
2021-01-07  39.239677  38.422000  38.448099  18218800  
2021-01-08  39.500638  38.491593  38.691662  20936300  

[5 rows x 70 columns]

In [9]:
assets = df.columns.get_level_values(0).unique()
assets

Index(['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO',
       'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'],
      dtype='object')

In [10]:
time_prd = 20
processed_dfs = []
day_shift = 1

for symbol in assets:
    asset_df = df.xs(symbol, level=0, axis=1).copy()
    for target in targets:
        s = asset_df[target]

        # Calc Log Return 
        asset_df[f"Log_Returns {target}"] = np.log(s).diff()
        
        asset_df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd).shift(day_shift)
        asset_df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd).shift(day_shift)
        asset_df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd).shift(day_shift)
    asset_df.columns = pd.MultiIndex.from_product([[symbol], asset_df.columns])
    processed_dfs.append(asset_df)
    
df_updated = pd.concat(processed_dfs, axis=1)
df_updated.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                                                                     \
           Log_Returns Close SMA_20 Close EMA_20 Close RSI_20 Close   
Date                                                                  
2021-01-04               NaN          NaN          NaN          NaN   
2021-01-05          0.012288          NaN          NaN          NaN   
2021-01-06         -0.034241          NaN          NaN          NaN   
2021-01-07          0.033554          NaN          NaN          NaN   
2021-01-08          0.008594          NaN          NaN          NaN   

                  TSLA  ...          AMD       CSCO                        \
                 Close  ... RSI_20 Close      Close       High        Low   
Date                    ...                                                 
2021-01-04  243.256668  ...          NaN  38.239326  38.595972  37.708707   
2021-01-05  245.036667  ...          NaN  38.256729  38.335017  37.734811   
2021-01-06  251.993332  ...          NaN  38.622074  39.030909  38.178440   
2021-01-07  272.013336  ...          NaN  39.109196  39.239677  38.422000   
2021-01-08  293.339996  ...          NaN  39.196186  39.500638  38.491593   

                                                                             \
                 Open    Volume Log_Returns Close SMA_20 Close EMA_20 Close   
Date                                                                          
2021-01-04  38.543782  24392500               NaN          NaN          NaN   
2021-01-05  37.995770  17763700          0.000455          NaN          NaN   
2021-01-06  38.387210  21823100          0.009505          NaN          NaN   
2021-01-07  38.448099  18218800          0.012534          NaN          NaN   
2021-01-08  38.691662  20936300          0.002222          NaN          NaN   

                         
           RSI_20 Close  
Date                     
2021-01-04          NaN  
2021-01-05          NaN  
2021-01-06          NaN  
2021-01-07          NaN  
2021-01-08          NaN  

[5 rows x 126 columns]

In [11]:
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])

AAPL   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
TSLA   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
MSFT   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
NVDA   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
GOOGL  Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
AMZN   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
GOOG   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
META   Log_Returns Close     1
       SMA_20 Close         20
       EMA_20 Close         20
       RSI_20 Close         21
AVGO   L

In [12]:
df_updated.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_updated

Series([], dtype: int64)


AAPL                                                \
                 Close        High         Low        Open    Volume   
Date                                                                   
2021-02-03  130.510605  132.293751  130.189052  132.283998  89880900   
2021-02-04  133.872253  133.881992  131.143942  132.810165  84183100   
2021-02-05  133.457504  134.101570  132.579243  134.033268  75693800   
2021-02-08  133.603882  133.652677  131.661931  132.745127  71297200   
2021-02-09  132.725632  134.550485  132.569507  133.320902  76774200   
...                ...         ...         ...         ...       ...   
2024-12-24  257.286682  257.296626  254.386957  254.586262  23234700   
2024-12-26  258.103729  259.179926  256.718662  257.276679  27237100   
2024-12-27  254.685867  257.784882  252.164818  256.917934  42355300   
2024-12-30  251.307877  252.603281  249.863009  251.337769  35557500   
2024-12-31  249.534180  252.384064  248.547676  251.547039  39480700   

                                                                     \
           Log_Returns Close SMA_20 Close EMA_20 Close RSI_20 Close   
Date                                                                  
2021-02-03         -0.007809   129.956166   129.860444    55.492134   
2021-02-04          0.025432   130.098915   129.922364    54.310490   
2021-02-05         -0.003103   130.624602   130.298544    57.444593   
2021-02-08          0.001096   130.919080   130.599397    56.937364   
2021-02-09         -0.006595   131.165826   130.885538    57.078163   
...                      ...          ...          ...          ...   
2024-12-24          0.011413   244.160249   244.586372    70.294542   
2024-12-26          0.003171   245.422270   245.795925    72.409437   
2024-12-27         -0.013331   246.616031   246.968097    72.976182   
2024-12-30         -0.013352   247.645377   247.703123    66.922965   
2024-12-31         -0.007083   248.386246   248.046433    61.606624   

                  TSLA  ...          AMD       CSCO                        \
                 Close  ... RSI_20 Close      Close       High        Low   
Date                    ...                                                 
2021-02-03  284.896667  ...    45.723516  39.813793  40.153041  39.613724   
2021-02-04  283.329987  ...    44.591484  41.101185  41.162075  39.813783   
2021-02-05  284.076660  ...    44.531658  41.823177  42.049341  41.318653   
2021-02-08  287.806671  ...    44.625504  42.571262  42.919210  42.240716   
2021-02-09  283.153320  ...    49.930654  42.188519  42.475576  42.110230   
...                ...  ...          ...        ...        ...        ...   
2024-12-24  462.279999  ...    38.503501  58.345390  58.345390  57.321788   
2024-12-26  454.130005  ...    40.465569  58.472118  58.550109  57.906701   
2024-12-27  431.660004  ...    39.500059  58.111423  58.511116  57.653238   
2024-12-30  417.410004  ...    39.660238  57.701981  57.896953  56.941591   
2024-12-31  403.839996  ...    37.452344  57.711735  57.887210  57.292545   

                                                                             \
                 Open    Volume Log_Returns Close SMA_20 Close EMA_20 Close   
Date                                                                          
2021-02-03  39.796395  13173600         -0.001310    39.276211    39.258793   
2021-02-04  39.900768  22285700          0.031824    39.354064    39.311650   
2021-02-05  41.379544  25488600          0.017414    39.478020    39.482082   
2021-02-08  42.240716  25215400          0.017729    39.613719    39.705043   
2021-02-09  42.423383  24795000         -0.009031    39.782473    39.978016   
...               ...       ...               ...          ...          ...   
2024-12-24  57.321788   9922300          0.014643    57.445106    57.009285   
2024-12-26  58.121168   8524500          0.002170    57.499211    57.136533   
2024-12-27  58.072428  13021400         -0.006188    57.518220    

In [13]:
drop_cols = ['Open', 'High', 'Low', 'Volume']

df_final = df_updated.drop(columns=drop_cols, level=1).copy()
df_final

AAPL                                              \
                 Close Log_Returns Close SMA_20 Close EMA_20 Close   
Date                                                                 
2021-02-03  130.510605         -0.007809   129.956166   129.860444   
2021-02-04  133.872253          0.025432   130.098915   129.922364   
2021-02-05  133.457504         -0.003103   130.624602   130.298544   
2021-02-08  133.603882          0.001096   130.919080   130.599397   
2021-02-09  132.725632         -0.006595   131.165826   130.885538   
...                ...               ...          ...          ...   
2024-12-24  257.286682          0.011413   244.160249   244.586372   
2024-12-26  258.103729          0.003171   245.422270   245.795925   
2024-12-27  254.685867         -0.013331   246.616031   246.968097   
2024-12-30  251.307877         -0.013352   247.645377   247.703123   
2024-12-31  249.534180         -0.007083   248.386246   248.046433   

                               TSLA                                 \
           RSI_20 Close       Close Log_Returns Close SMA_20 Close   
Date                                                                 
2021-02-03    55.492134  284.896667         -0.020956   278.826500   
2021-02-04    54.310490  283.329987         -0.005514   280.819500   
2021-02-05    57.444593  284.076660          0.002632   282.386333   
2021-02-08    56.937364  287.806671          0.013045   282.989499   
2021-02-09    57.078163  283.153320         -0.016300   282.712833   
...                 ...         ...               ...          ...   
2024-12-24    70.294542  462.279999          0.070991   396.037001   
2024-12-26    72.409437  454.130005         -0.017787   402.221501   
2024-12-27    72.976182  431.660004         -0.050745   408.016501   
2024-12-30    66.922965  417.410004         -0.033569   412.955000   
2024-12-31    61.606624  403.839996         -0.033050   416.567500   

                                      ...         AMD                    \
           EMA_20 Close RSI_20 Close  ...       Close Log_Returns Close   
Date                                  ...                                 
2021-02-03   277.822563    63.563592  ...   87.889999         -0.010976   
2021-02-04   278.496287    61.346656  ...   87.839996         -0.000569   
2021-02-05   278.956640    60.767340  ...   87.900002          0.000683   
2021-02-08   279.444261    60.952329  ...   91.470001          0.039811   
2021-02-09   280.240681    61.897070  ...   90.910004         -0.006141   
...                 ...          ...  ...         ...               ...   
2024-12-24   401.852614    64.176147  ...  126.290001          0.013472   
2024-12-26   407.607603    68.373812  ...  125.059998         -0.009787   
2024-12-27   412.038308    66.270966  ...  125.190002          0.001039   
2024-12-30   413.907041    60.840552  ...  122.440002         -0.022211   
2024-12-31   414.240657    57.685101  ...  120.790001         -0.013568   

                                                        CSCO  \
           SMA_20 Close EMA_20 Close RSI_20 Close      Close   
Date                                                           
2021-02-03    91.305500    91.228214    45.723516  39.813793   
2021-02-04    91.061500    90.910289    44.591484  41.101185   
2021-02-05    90.937000    90.617880    44.531658  41.823177   
2021-02-08    90.573999    90.359035    44.625504  42.571262   
2021-02-09    90.418499    90.464841    49.930654  42.188519   
...                 ...          ...          ...        ...   
2024-12-24   132.116500   130.251952    38.503501  58.345390   
2024-12-26   131.374500   129.874623    40.465569  58.472118   
2024-12-27   130.741499   129.416088    39.500059  58.111423   
2024-12-30   130.188999   129.013603    39.660238  57.701981   
2024-12-31   129.452000   128.387546    37.452344  57.711735   

                                                                     
           Log_Returns Close SMA_20 Cl

In [14]:
def create_sequences(df, feature_name, steps: int):
    target_data = df.xs(feature_name, level=1, axis=1)
    return (target_data)

df_series_seq= create_sequences(df_final, "Close", 8)
df_series_seq

,AAPL,TSLA,MSFT,NVDA,GOOGL,AMZN,GOOG,META,AVGO,ORCL,CRM,ADBE,AMD,CSCO
Date,,,,,,,,,,,,,,
2021-02-03,130.510605,284.896667,233.604568,13.493309,102.172020,165.626495,102.800018,264.800354,41.928913,58.210693,232.375610,481.920013,87.889999,39.813793
2021-02-04,133.872253,283.329987,232.652863,13.626689,101.911491,166.550003,102.417633,264.641357,42.419239,59.315540,235.502716,489.380005,87.839996,41.101185
2021-02-05,133.457504,284.076660,232.835480,13.553641,103.658287,167.607498,104.187027,266.240204,42.002819,59.549629,236.403259,492.119995,87.900002,41.823177
2021-02-08,133.603882,287.806671,233.095016,14.399062,103.444397,166.147003,103.934258,264.730743,42.605808,59.090828,236.442825,493.760010,91.470001,42.571262
2021-02-09,132.725632,283.153320,234.344788,14.224045,102.991325,165.250000,103.467445,267.580872,42.779770,59.615166,234.236038,496.049988,90.910004,42.188519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,462.279999,436.929138,140.189468,195.344940,229.050003,196.932236,605.839600,237.988037,169.721893,342.748352,447.940002,126.290001,58.345390
2024-12-26,258.103729,454.130005,435.715790,139.899521,194.836945,227.050003,196.463745,601.453369,243.627945,169.989227,340.051605,450.160004,125.059998,58.472118
2024-12-27,254.685867,431.660004,428.177216,136.980164,192.007996,223.750000,193.413620,597.924561,240.043442,167.296021,336.797607,446.480011,125.190002,58.111423


In [14]:
close_price_df = df_final.xs('Close', level=1, axis=1)
close_price_df

,AAPL,TSLA,MSFT,NVDA,GOOGL,AMZN,GOOG,META,AVGO,ORCL,CRM,ADBE,AMD,CSCO
Date,,,,,,,,,,,,,,
2021-02-03,130.510605,284.896667,233.604568,13.493309,102.172020,165.626495,102.800018,264.800354,41.928913,58.210693,232.375610,481.920013,87.889999,39.813793
2021-02-04,133.872253,283.329987,232.652863,13.626689,101.911491,166.550003,102.417633,264.641357,42.419239,59.315540,235.502716,489.380005,87.839996,41.101185
2021-02-05,133.457504,284.076660,232.835480,13.553641,103.658287,167.607498,104.187027,266.240204,42.002819,59.549629,236.403259,492.119995,87.900002,41.823177
2021-02-08,133.603882,287.806671,233.095016,14.399062,103.444397,166.147003,103.934258,264.730743,42.605808,59.090828,236.442825,493.760010,91.470001,42.571262
2021-02-09,132.725632,283.153320,234.344788,14.224045,102.991325,165.250000,103.467445,267.580872,42.779770,59.615166,234.236038,496.049988,90.910004,42.188519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,462.279999,436.929138,140.189468,195.344940,229.050003,196.932236,605.839600,237.988037,169.721893,342.748352,447.940002,126.290001,58.345390
2024-12-26,258.103729,454.130005,435.715790,139.899521,194.836945,227.050003,196.463745,601.453369,243.627945,169.989227,340.051605,450.160004,125.059998,58.472118
2024-12-27,254.685867,431.660004,428.177216,136.980164,192.007996,223.750000,193.413620,597.924561,240.043442,167.296021,336.797607,446.480011,125.190002,58.111423


In [15]:
df_final.columns.remove_unused_levels()

n_obs = len(df_final)
n_assets = len(df_final.columns.get_level_values(0).unique())
n_features = len(df_final.columns.get_level_values(1).unique())

print(n_obs, n_assets, n_features)

market = df_final.values.reshape(n_obs, n_assets, n_features)
print(type(market))
market.shape

# tensor_list = []
     #    for symbol in self.symbols:
     #        if symbol in self.assets:
     #            # Call Asset method
     #            asset_tensor = self.assets[symbol].to_tensor(features, device)
     #            tensor_list.append(asset_tensor)
        
     #    # Stack along dimension 1 (Dimension N)
     #    # Asset: [T, F] -> Stack dim=1 -> [T, A, F]
     #    basket_tensor = torch.stack(tensor_list, dim=1)

984 14 5
<class 'numpy.ndarray'>


(984, 14, 5)

In [16]:
df_final.columns.levels[1]

Index(['Close', 'EMA_20 Close', 'High', 'Log_Returns Close', 'Low', 'Open',
       'RSI_20 Close', 'SMA_20 Close', 'Volume'],
      dtype='object')

In [17]:
n_prices = 1
n_targets = 1
# market[0, 0, 0] # -> that close price 

print(f"Market shape:\t\t{market.shape}")
print(f"Close price:\t\t{market[:,:,0:1].shape}")
print(f"Features_Target:\t{market[:,:,1:n_targets + 1].shape}")
print(f"Features_Condition:\t{market[:,:,n_targets + 1:].shape}")

print(f"That's a close price:\t{market[0,0,0:1]}")
print(f"Feature Target values:\t{market[0, 0, 1:n_targets + 1]}")
print(f"Feature_Cond values:\t{market[0, 0, n_targets + 1:]}")

price = market[:, :, 0:1]
x     = market[:, :, 1: n_targets + 1]
cond  = market[:, :, n_targets + 1:]
date  = df_final.index.get_level_values(0).unique().to_numpy()

print(f"price shape: {price.shape}")
print(f"x shape: {x.shape}")
print(f"cond shape: {cond.shape}")
print(f"date shape: {date.shape}")

Market shape:		(984, 14, 5)
Close price:		(984, 14, 1)
Features_Target:	(984, 14, 1)
Features_Condition:	(984, 14, 3)
That's a close price:	[130.51060486]
Feature Target values:	[-0.00780877]
Feature_Cond values:	[129.9561657  129.86044357  55.4921339 ]
price shape: (984, 14, 1)
x shape: (984, 14, 1)
cond shape: (984, 14, 3)
date shape: (984,)


In [18]:
ratios = [0.8, 0.1, 0.1]
total_count = len(market)
train_count = int(total_count * ratios[0])
val_count = int(total_count * ratios[1])
test_count = total_count - train_count - val_count

print(f"Ratios DS\nTrain:\t{train_count}\nVal:\t{val_count}\nTest:\t{test_count}\nTotal:\t{total_count}")

Ratios DS
Train:	787
Val:	98
Test:	99
Total:	984


In [19]:
all_dates = df_final.index.get_level_values(0).unique().to_numpy()
print(f"Dates shape: {all_dates.shape}")

Dates shape: (984,)


In [20]:
all_prices = market[:, :, 0]
# market = market[:, :, 1:]

print(f"Prices shape: {all_prices.shape}")
print(f"Market shape: {market.shape}")

Prices shape: (984, 14)
Market shape: (984, 14, 5)


In [21]:
end_val = train_count + val_count
train_part = {
    "x": x[:train_count],
    "cond": cond[:train_count],
    "date": date[:train_count],
    "price": price[:train_count],
}

val_part = {
    "x": x[train_count:end_val],
    "cond": cond[train_count:end_val],
    "date": date[train_count:end_val],
    "price": price[train_count:end_val],
}

test_part = {
    "x": x[end_val:],
    "cond": cond[end_val:],
    "date": date[end_val:],
    "price": price[end_val:],
}

print(f"Train X: {train_part['x'].shape}\nVal X: {val_part['x'].shape}\nTest X:{test_part['x'].shape}")
print(f"-" * 30)
print(f"Train Cond: {train_part['cond'].shape}\nVal Cond: {val_part['cond'].shape}\nTest Cond:{test_part['cond'].shape}")
print(f"-" * 30)
print(f"Train Price: {train_part['price'].shape}\nVal Price: {val_part['price'].shape}\nTest Price:{test_part['price'].shape}")
print(f"-" * 30)
print(f"Train Dates: {train_part['date'].shape}\nVal Dates: {val_part['date'].shape}\nTest Dates:{test_part['date'].shape}")

Train X: (787, 14, 1)
Val X: (98, 14, 1)
Test X:(99, 14, 1)
------------------------------
Train Cond: (787, 14, 3)
Val Cond: (98, 14, 3)
Test Cond:(99, 14, 3)
------------------------------
Train Price: (787, 14, 1)
Val Price: (98, 14, 1)
Test Price:(99, 14, 1)
------------------------------
Train Dates: (787,)
Val Dates: (98,)
Test Dates:(99,)


In [22]:
scaler_x = StandardScaler()
scaler_cond = StandardScaler()

# Scale X
x_train_2d = train_part['x'].reshape(train_part['x'].shape[0], -1)
x_val_2d = val_part['x'].reshape(val_part['x'].shape[0], -1)
x_test_2d = test_part['x'].reshape(test_part['x'].shape[0], -1)

print(x_train_2d.shape, x_val_2d.shape, x_test_2d.shape)

scaler_x.fit(x_train_2d)

train_x_scaled = scaler_x.transform(x_train_2d).reshape(train_part['x'].shape)
val_x_scaled = scaler_x.transform(x_val_2d).reshape(val_part['x'].shape)
test_x_scaled = scaler_x.transform(x_test_2d).reshape(test_part['x'].shape)

inspect(train_x_scaled, "X_scaled - Train Part")
inspect(val_x_scaled, "X_scaled - Val Part")
inspect(test_x_scaled, "X_scaled - Test Part")

# Scale Cond
cond_train_2d = train_part['cond'].reshape(train_part['cond'].shape[0], -1)
cond_val_2d = val_part['cond'].reshape(val_part['cond'].shape[0], -1)
cond_test_2d = test_part['cond'].reshape(test_part['cond'].shape[0], -1)

scaler_cond.fit(cond_train_2d)

train_cond_scaled = scaler_cond.transform(cond_train_2d).reshape(train_part['cond'].shape)
val_cond_scaled = scaler_cond.transform(cond_val_2d).reshape(val_part['cond'].shape)
test_cond_scaled = scaler_cond.transform(cond_test_2d).reshape(test_part['cond'].shape)

inspect(train_cond_scaled, "Cond_scaled - Train Part")
inspect(val_cond_scaled, "Cond_scaled - Val Part")
inspect(test_cond_scaled, "Cond_scaled - Test Part")

(787, 14) (98, 14) (99, 14)
--- Inspecting: X_scaled - Train Part ---
------------------------------------
Shape: (787, 14, 1)
Min:   -10.0905
Max:   7.5274
Mean:  -0.0000
Std:   1.0000
------------------------------------
--- Inspecting: X_scaled - Val Part ---
------------------------------------
Shape: (98, 14, 1)
Min:   -9.7527
Max:   6.4804
Mean:  -0.0183
Std:   1.0028
------------------------------------
--- Inspecting: X_scaled - Test Part ---
------------------------------------
Shape: (99, 14, 1)
Min:   -6.2767
Max:   10.3857
Mean:  0.0605
Std:   0.9547
------------------------------------
--- Inspecting: Cond_scaled - Train Part ---
------------------------------------
Shape: (787, 14, 3)
Min:   -2.7591
Max:   3.9034
Mean:  -0.0000
Std:   1.0000
------------------------------------
--- Inspecting: Cond_scaled - Val Part ---
------------------------------------
Shape: (98, 14, 3)
Min:   -2.8495
Max:   6.8965
Mean:  1.3615
Std:   1.7491
------------------------------------
--- 

(np.float64(-2.339867760201573),
 np.float64(7.951743081182571),
 np.float64(1.9506293572565483),
 np.float64(2.049531942976176))

In [23]:
# Pipeline([('Scaler_X', scaler_x), ('Scaler_Cond', scaler_cond)])

In [24]:
train_part['date'].shape
train_part['price'].shape

(787, 14, 1)

In [25]:
train_ds = MarketDataset(x=train_x_scaled, cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride)
val_ds = MarketDataset(x=val_x_scaled, cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride)
test_ds = MarketDataset(x=test_x_scaled, cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tcond: {train_ds[0]['cond'].shape}\n\tdates: {len(train_ds[0]['date'])}")

Num of Windows
Train DS: 145, Val Ds: 7, Test DS: 8

A sample shape from Train DS
	x: (1, 64, 14),
	cond: (3, 64, 14)
	dates: 64


In [26]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size_exp, shuffle=False)

batch_train = next(iter(train_loader))
print(len(train_loader))
print(batch_train["x"].shape)
print(batch_train["cond"].shape)

5
torch.Size([32, 1, 64, 14])
torch.Size([32, 3, 64, 14])


In [27]:
len(test_loader)

8

In [28]:
next(iter(test_loader))["x"].shape

torch.Size([1, 1, 64, 14])

In [29]:
B, C_cond, T, A = batch_train["cond"].shape
B, C_target, T, A = batch_train["x"].shape
print(f"B: {B}, C_target: {C_target}, C_cond: {C_cond}, T: {T}, A: {A}")

B: 32, C_target: 1, C_cond: 3, T: 64, A: 14


## Fit Model with Best Params from Optuna

In [30]:
# ignore droupout
# lr 
# too high
import os
steps_sim = 16

cfg = TrainConfig(epochs=2, window_size=64, device=torch.device("cuda:1"))
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
batch_size_exp = 1
epochs = cfg.epochs
sim_steps = cfg.steps_to_sim
num_sims = cfg.num_sims

# Optimizer
weight_decay = cfg.optimizer.weight_decay
lr = cfg.optimizer.lr
stride = 5
time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 2,
    'dim_feedforward': 896,
    'dropout': 0.1
}

ckpt_name = f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}"



model = DiffusionTransformer(
    num_assets          = A,
    num_channels        = C_target,
    num_cond_channels   = C_cond,
    num_layers          = ddpm_transformer['num_layers'],
    num_attention_heads = ddpm_transformer['nhead'],
    seq_length          = T,
    d_model             = ddpm_transformer['d_model'],
    dim_feedforward     = ddpm_transformer['dim_feedforward'],
    dropout             = ddpm_transformer['dropout']
).to(device)

diffusion = Diffusion(
    model=model,
    timesteps=ddpm['timesteps'],
    beta_start=ddpm['beta_start'],
    beta_end=ddpm['beta_end']
).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)


engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device,
    checkpoint_dir  = os.path.join(RESULTS_DIR,"optuna_checkpoints")
)

input dim x: 14, d_model: 128


2026-02-11 18:07:34,350 - Engine - INFO - Engine initialized on cuda:1
2026-02-11 18:07:34,351 - Engine - INFO - Criterion: MSELoss


In [31]:
checkpoint_filename = f"best_tr50_10_d128_dff896_l2_h8_t1000.pt" # d = d_model, l = n_layers, h = head attention , t = timestep

In [32]:
engine.load_checkpoint(checkpoint_filename)

/home/narodom.y@FUSION.LAB/research/src/engine/trainer.py:171: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=self.device)
2026-02-11 18:

# Experimental

In [33]:
batch_test = next(iter(test_loader))
print(f"x: {batch_test['x'].shape}, x_cond: {batch_test['cond'].shape}, dates: {batch_test['date'].shape}, prices: {batch_test['price'].shape}")

x: torch.Size([1, 1, 64, 14]), x_cond: torch.Size([1, 3, 64, 14]), dates: torch.Size([1, 64]), prices: torch.Size([1, 1, 64, 14])


In [34]:
def transform_dates(dates: torch.Tensor) -> pd.DatetimeIndex:
    dates_np = dates.cpu().numpy()
    dates_pd = pd.to_datetime(dates_np.flatten(), unit='ns')
    return dates_pd

def expand_mc_size(x: torch.Tensor, cond: torch.Tensor, n_samples: int):
    return x.expand(n_samples, -1, -1, -1), cond.expand(n_samples, -1, -1, -1)
    
# _, sim_genai = engine.simulate(batch_test['x'].to(device), batch_test['cond'].to(device), steps=8)
# sim_genai.shape

## Monte Carlo

### Monte Carlo GenAI

In [35]:
x_mc, cond_mc = expand_mc_size(batch_test['x'], batch_test['cond'], 1000)
print(x_mc.shape, cond_mc.shape)

torch.Size([1000, 1, 64, 14]) torch.Size([1000, 3, 64, 14])


In [36]:
_, sim_genai = engine.simulate(x_mc.to(device), cond_mc.to(device), steps=steps_sim)
sim_genai.shape

                                                              
KeyboardInterrupt



### Monte Carlo Statistic

In [ ]:
# [B, C, T, A]
print(batch_test['x'].shape, batch_test['cond'].shape)

x_blaf = batch_test['x'].permute(0, 2, 3, 1) # [B, T, A, C]
cond_blaf = batch_test['cond'].permute(0, 2, 3, 1) # [B, T, A, C]

x_laf = x_blaf.reshape(-1, x_blaf.shape[2], x_blaf.shape[3])
cond_laf = cond_blaf.reshape(-1, cond_blaf.shape[2], cond_blaf.shape[3])

print(x_laf.shape, cond_laf.shape)

x_2d = x_laf.reshape(x_laf.shape[0], -1)
cond_2d = cond_laf.reshape(cond_laf.shape[0], -1)

print(x_2d.shape, cond_2d.shape)

# Require [Length, Assets, Feature]

x_unscaled = scaler_x.inverse_transform(x_2d).reshape(x_laf.shape)
cond_unscaled = scaler_cond.inverse_transform(cond_2d).reshape(cond_laf.shape)

print(x_unscaled.shape, cond_unscaled.shape)

In [ ]:
B = batch_test['x'].shape[0]
T = batch_test['x'].shape[2]

BT, A, C = x_unscaled.shape
x_unscaled = x_unscaled.reshape(B, T, A, C).transpose(0, 3, 1, 2)

print(x_unscaled.shape)

In [ ]:
B = batch_test['cond'].shape[0]
T = batch_test['cond'].shape[2]

BT, A, C = cond_unscaled.shape
cond_unscaled = cond_unscaled.reshape(B, T, A, C).transpose(0, 3, 1, 2)

print(cond_unscaled.shape)

In [ ]:
type(x_unscaled)

In [ ]:
# [B, C, L, A]
# C = 1. Close log Return
#     ...
x_unscaled_picked = x_unscaled[0, 0, :, :]
print(x_unscaled_picked.shape)
sim_stat = monte_carlo_statistic(torch.as_tensor(x_unscaled_picked), n_sims=1000, steps=steps_sim) # [n_sims, steps, assets]
sim_stat.shape

In [ ]:
print(sim_genai.shape, sim_stat.shape)

In [ ]:
sim_genai_picked = sim_genai[:, 0, :, :]
print(sim_genai_picked.shape, sim_stat.shape)

In [ ]:
gt = x_unscaled_picked
gt.shape

In [ ]:
dates = transform_dates(batch_test['date'])
dates.shape

In [ ]:
risk_free_rate = 0.02/252
weight_bounds = (0, 1)
is_calc_distribution = True
report_filename = "Hello_World"

portfolio = Portfolio(risk_free_rate, weight_bounds=weight_bounds, save_dir=REPORTS_QS_DIR)

if is_calc_distribution:
    mu_genai, sigma_genai = portfolio.calc_distribution(sim_genai_picked.tolist())
    mu_stats, sigma_stats = portfolio.calc_distribution(sim_stat.tolist())
else:
    mu_genai, sigma_genai = portfolio.calc(sim_genai_picked)
    mu_stats, sigma_stats = portfolio.calc(sim_stat)

print(f"Mu_genai: {mu_genai.shape}, Sigma_genai: {sigma_genai.shape}")
print(f"Mu_stats: {mu_stats.shape}, Sigma_stats: {sigma_stats.shape}")
# logger.debug(f"Mu_genai: {mu_genai.shape}, Sigma_genai: {sigma_genai.shape}")
# logger.debug(f"Mu_stats: {mu_stats.shape}, Sigma_stats: {sigma_stats.shape}")

weights_genai = portfolio.optimize_weights(mu_genai, sigma_genai, risk_free_rate=risk_free_rate, scipy=True)
weights_stats = portfolio.optimize_weights(mu_stats, sigma_stats, risk_free_rate=risk_free_rate, scipy=True)

print(f"weights_genai: {weights_genai.shape}")
print(f"weights_stats: {weights_stats.shape}")
# logger.debug(f"weights_genai: {weights_genai.shape}")
# logger.debug(f"weights_stats: {weights_stats.shape}")

report = portfolio.back_test(weights=weights_genai, weights_benchmark=weights_stats, returns=gt, dates=dates, is_saved=True, filename=report_filename)